https://huggingface.co/spaces/sdadas/pirb

### Sparse:
 - gm-jaro/sdadas-polish-splade-onnx

### Dense:
 - sdadas/stella-pl-retrieval-8k
 - sdadas/stella-pl-retrieval-mini-8k
 - ipipan/silver-retriever-base-v1.1
 - sdadas/stella-pl-retrieval
 - sdadas/mmlw-retrieval-roberta-large-v2

### Reranker:
 - sdadas/polish-reranker-roberta-v2
 - sdadas/polish-reranker-roberta-v3

In [ ]:
# Idea: https://github.com/qdrant/fastembed/issues/294
from fastembed.common.model_description import SparseModelDescription, ModelSource
from fastembed.sparse.splade_pp import supported_splade_models

supported_splade_models.append(SparseModelDescription(
        model="gm-jaro/sdadas-polish-splade-onnx",
        vocab_size=50001,
        description="Polish export from sdadas-polish-splade to onnx.",
        license="apache-2.0",
        size_in_GB=0.325,
        sources=ModelSource(hf="gm-jaro/sdadas-polish-splade-onnx"),
        model_file="model.onnx",
    ))

In [ ]:
from qdrant_client import QdrantClient, models
from fastembed import SparseTextEmbedding

MODEL_NAME = "gm-jaro/sdadas-polish-splade-onnx"

print(f"Initializing model: {MODEL_NAME}...")
model = SparseTextEmbedding(model_name=MODEL_NAME)


client = QdrantClient(":memory:")
COLLECTION_NAME = "test_sparse_collection"

client.create_collection(
    collection_name=COLLECTION_NAME,
    sparse_vectors_config={
        "polish-splade": models.SparseVectorParams()
    }
)


documents = [
    "Warszawa to stolica Polski i największe miasto w kraju.",
    "Góry Tatry są najwyższym pasmem górskim w Polsce.",
    "Lubię spacerować po Tatrach.",
    "Lubię spacerować po Tatrach - najwyższych górach w Polsce.",
    "Pierogi to tradycyjne polskie danie, które każdy uwielbia.",
    "Programowanie w Pythonie jest przyjemne i efektywne."
]


embeddings = list(model.embed(documents))


print("Indexing documents...")
client.upsert(
    collection_name=COLLECTION_NAME,
    points=[
        models.PointStruct(
            id=idx,
            payload={"text": doc},
            vector={"polish-splade": emb.as_object()}
        )
        for idx, (doc, emb) in enumerate(zip(documents, embeddings))
    ]
)


query = "Jakie są najwyższe góry w Polsce?"
query_vector = list(model.embed([query]))[0]

print(f"\nQuery: {query}")

sparse_query = models.SparseVector(
    indices=query_vector.indices.tolist(),
    values=query_vector.values.tolist(),
)

search_result = client.query_points(
    collection_name=COLLECTION_NAME,
    query=sparse_query,
    using="polish-splade",
    limit=10,
    with_payload=True,
)

print("\nSearch results:")
for res in search_result.points:
    print(f" Score: {res.score:.4f} | Text: {res.payload['text']}")

In [ ]:
# https://docs.langchain.com/oss/python/integrations/vectorstores/qdrant#hybrid-vector-search
from langchain_qdrant import RetrievalMode
from langchain_qdrant import QdrantVectorStore
from qdrant_client.http.models import Distance, VectorParams, SparseVectorParams
from langchain_qdrant import FastEmbedSparse
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings


DENSE_MODEL_NAME = "sdadas/stella-pl-retrieval-mini-8k"
embeddings = HuggingFaceEmbeddings(model=DENSE_MODEL_NAME, model_kwargs={"trust_remote_code": True})
sparse_embeddings = FastEmbedSparse(model_name=MODEL_NAME)

client.create_collection(
    collection_name=COLLECTION_NAME + "hybrid",
    vectors_config={"dense": VectorParams(size=1024, distance=Distance.COSINE)},
    sparse_vectors_config={
        "sparse": SparseVectorParams(index=models.SparseIndexParams(on_disk=False))
    },
)

qdrant = QdrantVectorStore(
    client=client,
    collection_name=COLLECTION_NAME + "hybrid",
    embedding=embeddings,
    sparse_embedding=sparse_embeddings,
    retrieval_mode=RetrievalMode.HYBRID,
    vector_name="dense",
    sparse_vector_name="sparse",
)

docs = [
    Document(
        page_content=text,
        metadata={"id": idx, "source": "local"}
    )
    for idx, text in enumerate(documents)
]

qdrant.add_documents(documents=docs)

query = "Jakie są najwyższe góry w Polsce?"
found_docs = qdrant.similarity_search(query, k=10)

print("\nSearch results:")
for idx, res in enumerate(found_docs):
    print(f"res-{idx}: {res.page_content}")

In [ ]:
# https://qdrant.tech/documentation/advanced-tutorials/reranking-hybrid-search/
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_community.cross_encoders import HuggingFaceCrossEncoder


base_retriever = qdrant.as_retriever(search_kwargs={"k": 10})

RERANK_MODEL_NAME = "sdadas/polish-reranker-roberta-v3"
model = HuggingFaceCrossEncoder(model_name=RERANK_MODEL_NAME, model_kwargs={'trust_remote_code': True})
compressor = CrossEncoderReranker(model=model, top_n=5)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever
)

query = "Jakie są najwyższe góry w Polsce?"
found_docs = compression_retriever.invoke(query)

print("\nResults after reranking:")
for idx, res in enumerate(found_docs):
    print(f"res-{idx}: {res.page_content}")

# TODO: remove the Hugging Face cache